# Hyperparameter Tuning and Leave-One-Pathogen-Out Validation

## Overview

The analysis is organized into sequential sections, with each section performing a specific stage of the workflow. **The workflow is intentionally divided into sections because hyperparameter optimization can be computationally intensive and may require substantial time and memory.** This organization allows individual stages to be run independently, inspected, and resumed without repeating previously completed computationally expensive steps.

The overall workflow consists of the following stages:
1. **Configuration of the analysis** – Select the negative-to-positive interaction ratio and sequence feature representation to be analysed.
2. **Sequence and feature preparation** – Prepare the host and pathogen protein sequence features for machine learning.
3. **Construction of training-only cross-validation folds** – Generate predefined cross-validation folds from the earlier saved CV folds. These folds are used exclusively for model selection and hyperparameter optimization.
4. **Hyperparameter optimization** – Perform randomized hyperparameter searches for Random Forest, XGBoost, and LightGBM using the predefined cross-validation folds. The Matthews correlation coefficient (MCC) is used as the optimization criterion.
5. **Model selection** – Compare the optimized models based on their cross-validation MCC and select the best-performing model for the specified data ratio and feature representation.
6. **Final hold-out evaluation** – Refit the selected model using the complete training set and evaluate it once on the independent, locked test set. The test set is not used during cross-validation or hyperparameter tuning.
7. **Host-disjoint Leave-One-Pathogen-Out (LOPO) validation** – Evaluate transferability across the pathogen groups represented in the dataset by holding out one pathogen group at a time while preventing host-protein overlap between the training and held-out data.
8. **Performance visualization and result saving** – Generate confusion matrices, performance summaries, ROC curves, and LOPO performance plots, while saving trained models and result tables for reproducibility.
9. **Optional checkpoint-based analysis** – Previously saved models and results can be reloaded to perform LOPO evaluation without repeating the computationally intensive hyperparameter optimization stage.
10. **Deployment model refit** – Refit the selected best model on the complete dataset (training set + hold-out test set combined) to produce a deployment-ready model.
11. **Prediction on new HP-PPI pairs** – Load the saved deployment model and apply it to new, previously unlabeled host-pathogen pairs that have already been preprocessed and feature-extracted using the same representation as the rest of the pipeline, producing predicted interaction labels and probabilities.

###
For the additional LOPO analysis, the optimized hyperparameters obtained during the training stage are retained. The model is refitted independently for each LOPO iteration using the corresponding LOPO training data and evaluated on the held-out pathogen group. **No additional hyperparameter tuning is performed within the LOPO test set.**

### Computational Organization

Because the complete workflow can be computationally demanding, particularly when multiple feature representations, data ratios, models, and hyperparameter combinations are analysed, the notebook is designed as a **modular and resumable workflow**. Intermediate models, tuning summaries, datasets, and evaluation results are saved to disk. This means that a completed computational stage does not need to be repeated when proceeding to subsequent stages or when analysing another feature representation or interaction ratio.

To reproduce an analysis, the user can select the desired interaction ratio and feature representation in the configuration section and then execute the workflow sequentially. Alternatively, previously saved outputs can be loaded where available, allowing computationally expensive stages, such as hyperparameter optimisation, to be skipped.

Similarly, the deployment model (stage 10) and new-pair prediction (stage 11) can each be run independently in a fresh session, as long as the required upstream saved files (tuned model, or best-model selection + train/test features) are available on disk.

In [ ]:
#Mounting Google Drive to access files
import os
from google.colab import drive
drive.mount ('/content/my_drive')

#import pandas for data manipulation
import pandas as pd


##Section 0:
###**Set which ratio you're running (change this and rerun sections 2 onward per ratio)**

In [ ]:
# =============================================================================
# SECTION 0: SELECT CURRENT RATIO AND FEATURE SET
# =============================================================================
# Change CURRENT_RATIO to 1, 5, or 10 and rerun from Section 2 onward.
# This keeps each ratio's run independent and resumable.

CURRENT_RATIO = 1   # <-- change this per run: 1, 5, or 10
feature_name = "AAC"

base_hpi_path = "/content/my_drive/MyDrive/HPI"

ratio_config = {
    1:  {"features_dir": f"{base_hpi_path}/Features/Balanced", "ml_dir": f"{base_hpi_path}/ML/Balanced", "label": "1:1"},
    5:  {"features_dir": f"{base_hpi_path}/Features/R5",       "ml_dir": f"{base_hpi_path}/ML/R5",       "label": "1:5"},
    10: {"features_dir": f"{base_hpi_path}/Features/R10",      "ml_dir": f"{base_hpi_path}/ML/R10",      "label": "1:10"},
}

cfg = ratio_config[CURRENT_RATIO]
features_dir = cfg["features_dir"]
ml_dir = cfg["ml_dir"]
label_tag = cfg["label"].replace(":", "_")
os.makedirs(ml_dir, exist_ok=True)
os.chdir(ml_dir)

print(f"Ratio: {cfg['label']} | Features: {feature_name} | ml_dir: {ml_dir}")

##Section 1:
###All function definitions (run once per session, no computation)

In [ ]:
# =============================================================================
# SECTION 1: FUNCTION DEFINITIONS (run once per kernel session)
# =============================================================================

import os
import copy
import joblib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import RandomizedSearchCV, PredefinedSplit
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, matthews_corrcoef,
    roc_auc_score, confusion_matrix, roc_curve
)
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

non_feature_cols = [
    "host_sequence", "pathogen_sequence",
    "host_protein_id", "pathogen_protein_id",
    "host_cluster", "pathogen_cluster",
    "pathogen_specie", "host_partition",
    "pathogen_partition", "label",
    "_fold_id", "_cv_fold",
]

def get_X_y(df):
    feature_cols = [c for c in df.columns if c not in non_feature_cols]
    X = df[feature_cols].astype(float)
    y = df["label"].astype(int)
    return X, y

def create_predefined_cv(cv_folds_dir, feature_name, n_folds=10):
    fold_dfs = []
    for fold in range(n_folds):
        fold_file = f"{cv_folds_dir}/fold{fold}_val_{feature_name}_features.csv"
        fold_val = pd.read_csv(fold_file).copy()
        fold_val["_cv_fold"] = fold
        fold_dfs.append(fold_val)
    combined_df = pd.concat(fold_dfs, ignore_index=True)
    X, y = get_X_y(combined_df)
    assert "_cv_fold" not in X.columns, "_cv_fold leaked into features!"
    test_fold = combined_df["_cv_fold"].values
    cv = PredefinedSplit(test_fold=test_fold)
    return X, y, cv

MODEL_CONSTRUCTORS = {
    "RandomForest": lambda: RandomForestClassifier(random_state=42, n_jobs=-1),
    "XGBoost": lambda: XGBClassifier(eval_metric="logloss", random_state=42, n_jobs=-1),
    "LightGBM": lambda: LGBMClassifier(random_state=42, n_jobs=-1, verbosity=-1),
}

param_distributions = {
    "RandomForest": {
        "n_estimators": [200, 500, 1000],
        "max_depth": [None, 10, 20, 30],
        "min_samples_split": [2, 5, 10],
        "min_samples_leaf": [1, 2, 4],
        "max_features": ["sqrt", "log2"],
    },
    "XGBoost": {
        "n_estimators": [200, 300, 500],
        "max_depth": [3, 5, 7, 10],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
        "min_child_weight": [1, 3, 5],
    },
    "LightGBM": {
        "n_estimators": [200, 300, 500],
        "num_leaves": [31, 63, 127],
        "max_depth": [-1, 10, 20],
        "learning_rate": [0.01, 0.05, 0.1],
        "subsample": [0.7, 0.9, 1.0],
        "colsample_bytree": [0.7, 0.9, 1.0],
        "min_child_samples": [10, 20, 30],
    },
}

def tune_one_model(model_name, X, y, cv, n_iter=30):
    """Tunes a SINGLE model and returns (best_estimator, summary_row)."""
    model = MODEL_CONSTRUCTORS[model_name]()
    print(f"\n{'-'*70}\nTuning {model_name}\n{'-'*70}")

    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_distributions[model_name],
        n_iter=n_iter,
        scoring="matthews_corrcoef",
        cv=cv,
        random_state=42,
        n_jobs=-1,
        refit=True,
        verbose=1,
        return_train_score=True,
    )
    search.fit(X, y)

    print(f"\n{model_name} — Best CV MCC: {search.best_score_:.4f}")
    print(f"Best parameters: {search.best_params_}")

    summary_row = {
        "Model": model_name,
        "Best_CV_MCC": search.best_score_,
        "Best_Parameters": str(search.best_params_),
    }
    return search.best_estimator_, summary_row

def calculate_metrics(y_true, y_pred, y_prob):
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return {
        "Accuracy": accuracy_score(y_true, y_pred),
        "Sensitivity": recall_score(y_true, y_pred),
        "Specificity": specificity,
        "F1 Score": f1_score(y_true, y_pred),
        "MCC": matthews_corrcoef(y_true, y_pred),
        "AUC-ROC": roc_auc_score(y_true, y_prob),
    }

def evaluate_final_test(best_model_name, best_model, X_train, y_train, X_test, y_test):
    print(f"\nRetraining final model: {best_model_name}")
    best_model.fit(X_train, y_train)
    y_pred = best_model.predict(X_test)
    y_prob = best_model.predict_proba(X_test)[:, 1]
    metrics = calculate_metrics(y_test, y_pred, y_prob)
    results = {"Model": best_model_name, **metrics}
    return best_model, y_pred, y_prob, results

def evaluate_lopo_host_disjoint(features_dir, best_model_name, best_model, feature_name):
    lopo_results = []
    folder = "lopo_folds_host_disjoint"
    pathogen_names = {
        "Bacillus": "bacillus", "Escherichia coli": "ecoli",
        "Francisella": "francisella", "Yersinia": "yersinia",
    }
    for species, file_species in pathogen_names.items():
        train_file = f"{features_dir}/{folder}/lopo_{file_species}_train_{feature_name}_features.csv"
        test_file = f"{features_dir}/{folder}/lopo_{file_species}_test_{feature_name}_features.csv"
        if not os.path.exists(train_file) or not os.path.exists(test_file):
            print(f"Missing file(s) for {species}, skipping.")
            continue

        lopo_train = pd.read_csv(train_file)
        lopo_test = pd.read_csv(test_file)
        X_tr, y_tr = get_X_y(lopo_train)
        X_te, y_te = get_X_y(lopo_test)

        lopo_model = copy.deepcopy(best_model)
        lopo_model.fit(X_tr, y_tr)
        y_pred = lopo_model.predict(X_te)
        y_prob = lopo_model.predict_proba(X_te)[:, 1]
        metrics = calculate_metrics(y_te, y_pred, y_prob)

        lopo_results.append({"Model": best_model_name, "Held_Out_Pathogen": species, **metrics})
        print(f"{species}: MCC={metrics['MCC']:.3f}, AUC-ROC={metrics['AUC-ROC']:.3f}")

    return pd.DataFrame(lopo_results)

# --- plotting functions ---

def plot_final_test_confusion_matrix(y_test, y_pred, model_name, dataset_label, feature_name, save_path):
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=["Negative", "Positive"], yticklabels=["Negative", "Positive"],
                annot_kws={"size": 15})
    plt.xlabel("Predicted Class"); plt.ylabel("True Class")
    plt.title(f"Confusion Matrix — {dataset_label} ({feature_name})\nFinal Model: {model_name}")
    plt.tight_layout()
    plt.savefig(f"{save_path}.png", dpi=600, bbox_inches="tight")
    plt.close()
    return cm

def plot_final_test_bar_chart(results_df, title, save_path):
    metrics = ["Accuracy", "Sensitivity", "Specificity", "F1 Score", "MCC", "AUC-ROC"]
    model_name = results_df["Model"].iloc[0]
    values = [results_df[m].iloc[0] for m in metrics]
    plt.figure(figsize=(10, 6))
    bars = plt.bar(metrics, values)
    plt.ylim(0, 1); plt.ylabel("Score"); plt.title(f"{title}\nFinal Model: {model_name}")
    plt.xticks(rotation=30, ha="right")
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width()/2, value + 0.02, f"{value:.3f}", ha="center", fontsize=9)
    plt.tight_layout()
    plt.savefig(f"{save_path}.png", dpi=600, bbox_inches="tight")
    plt.close()

def plot_final_test_roc(y_test, y_prob, model_name, dataset_label, feature_name, save_path):
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)
    plt.figure(figsize=(7, 6))
    plt.plot(fpr, tpr, linewidth=2, label=f"{model_name} (AUC = {auc:.3f})")
    plt.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    plt.xlabel("False Positive Rate"); plt.ylabel("True Positive Rate")
    plt.title(f"ROC Curve — {dataset_label} ({feature_name})")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(f"{save_path}.png", dpi=600, bbox_inches="tight")
    plt.close()

def plot_lopo_results(lopo_results_df, title, save_path):
    metrics = ["Accuracy", "Sensitivity", "Specificity", "F1 Score", "MCC", "AUC-ROC"]
    pathogens = lopo_results_df["Held_Out_Pathogen"].tolist()
    x = np.arange(len(pathogens)); width = 0.12
    fig, ax = plt.subplots(figsize=(14, 7))
    for i, metric in enumerate(metrics):
        ax.bar(x + (i - 2.5) * width, lopo_results_df[metric].values, width, label=metric)
    ax.set_xlabel("Held-out pathogen"); ax.set_ylabel("Score"); ax.set_title(title)
    ax.set_xticks(x); ax.set_xticklabels(pathogens); ax.set_ylim(0, 1)
    ax.legend(bbox_to_anchor=(1.02, 1), loc="upper left")
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    plt.tight_layout()
    plt.savefig(f"{save_path}.png", dpi=600, bbox_inches="tight")
    plt.show(); plt.close()

def plot_lopo_heatmap(lopo_results_df, title, save_path):
    metrics = ["Accuracy", "Sensitivity", "Specificity", "F1 Score", "MCC", "AUC-ROC"]
    heatmap_df = lopo_results_df.set_index("Held_Out_Pathogen")[metrics]
    plt.figure(figsize=(10, 6))
    sns.heatmap(heatmap_df, annot=True, fmt=".3f", vmin=0, vmax=1, linewidths=0.5,
                cmap="Blues", cbar_kws={"label": "Score"})
    plt.title(title); plt.xlabel("Performance Metric"); plt.ylabel("Held-out pathogen")
    plt.tight_layout()
    plt.savefig(f"{save_path}.png", dpi=600, bbox_inches="tight")
    plt.close()

print("Section 1 complete: all functions defined.")

##Section 2: Build the CV split (run once per ratio)

In [ ]:
# =============================================================================
# SECTION 2: BUILD PREDEFINED CV SPLIT FOR THIS RATIO
# =============================================================================

cv_folds_dir = f"{features_dir}/cv_folds"
X_cv, y_cv, cv = create_predefined_cv(cv_folds_dir, feature_name, n_folds=10)
print(f"CV samples: {len(X_cv)}, features: {X_cv.shape[1]}")

##Section 3a: Tune RandomForest only — save immediately

In [ ]:
# =============================================================================
# SECTION 3a: TUNE RANDOM FOREST (checkpointed)
# =============================================================================

rf_model, rf_summary = tune_one_model("RandomForest", X_cv, y_cv, cv, n_iter=30)

joblib.dump(rf_model, f"{ml_dir}/{cfg['label'].replace(':','_')}_{feature_name}_tuned_RandomForest.pkl")
pd.DataFrame([rf_summary]).to_csv(f"{ml_dir}/{cfg['label'].replace(':','_')}_{feature_name}_tuning_RandomForest.csv", index=False)

print("RandomForest tuning saved.")

##Section 3b: Tune XGBoost only

In [ ]:
# =============================================================================
# SECTION 3b: TUNE XGBOOST (checkpointed)
# =============================================================================

xgb_model, xgb_summary = tune_one_model("XGBoost", X_cv, y_cv, cv, n_iter=30)

joblib.dump(xgb_model, f"{ml_dir}/{cfg['label'].replace(':','_')}_{feature_name}_tuned_XGBoost.pkl")
pd.DataFrame([xgb_summary]).to_csv(f"{ml_dir}/{cfg['label'].replace(':','_')}_{feature_name}_tuning_XGBoost.csv", index=False)

print("XGBoost tuning saved.")

##Section 3c: Tune LightGBM only

In [ ]:
# =============================================================================
# SECTION 3c: TUNE LIGHTGBM (checkpointed)
# =============================================================================

lgbm_model, lgbm_summary = tune_one_model("LightGBM", X_cv, y_cv, cv, n_iter=30)

joblib.dump(lgbm_model, f"{ml_dir}/{cfg['label'].replace(':','_')}_{feature_name}_tuned_LightGBM.pkl")
pd.DataFrame([lgbm_summary]).to_csv(f"{ml_dir}/{cfg['label'].replace(':','_')}_{feature_name}_tuning_LightGBM.csv", index=False)

print("LightGBM tuning saved.")

##Section 4: Reload all three tuning results, select the best model

This section is fully resumable independently — if your session restarted after Section 3c, you can jump straight here (after rerunning Section 0 and Section 1) without redoing any tuning.

In [ ]:
# =============================================================================
# SECTION 4: LOAD TUNING RESULTS, SELECT BEST MODEL
# =============================================================================

label_tag = cfg['label'].replace(':','_')

tuning_summaries = []
tuned_models = {}

for model_name in ["RandomForest", "XGBoost", "LightGBM"]:
    summary_path = f"{ml_dir}/{label_tag}_{feature_name}_tuning_{model_name}.csv"
    model_path = f"{ml_dir}/{label_tag}_{feature_name}_tuned_{model_name}.pkl"

    summary_df = pd.read_csv(summary_path)
    tuning_summaries.append(summary_df)
    tuned_models[model_name] = joblib.load(model_path)

tuning_results_df = pd.concat(tuning_summaries, ignore_index=True)
tuning_results_df.to_csv(f"{ml_dir}/{label_tag}_{feature_name}_hyperparameter_tuning_results.csv", index=False)
print(tuning_results_df)

best_idx = tuning_results_df["Best_CV_MCC"].idxmax()
best_model_name = tuning_results_df.loc[best_idx, "Model"]
best_model = tuned_models[best_model_name]

print(f"\nWINNING MODEL: {best_model_name} (CV MCC = {tuning_results_df.loc[best_idx, 'Best_CV_MCC']:.4f})")

best_model_selection_df = pd.DataFrame({
    "Dataset_Ratio": [cfg["label"]], "Feature": [feature_name],
    "Best_Model": [best_model_name], "Best_CV_MCC": [tuning_results_df.loc[best_idx, "Best_CV_MCC"]],
})
best_model_selection_df.to_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv", index=False)

##Section 5: Final test evaluation + LOPO + all visualizations

In [ ]:
# =============================================================================
# SECTION 5: FINAL TEST EVALUATION + LOPO + VISUALIZATIONS
# =============================================================================

train_df = pd.read_csv(f"{features_dir}/train_{feature_name}_split_features.csv")
test_df = pd.read_csv(f"{features_dir}/test_{feature_name}_split_features.csv")
X_train, y_train = get_X_y(train_df)
X_test, y_test = get_X_y(test_df)

final_model, y_test_pred, y_test_prob, test_metrics = evaluate_final_test(
    best_model_name, best_model, X_train, y_train, X_test, y_test
)

test_results_df = pd.DataFrame([test_metrics])
test_results_df.insert(0, "Dataset_Ratio", cfg["label"])
test_results_df.insert(1, "Feature", feature_name)
test_results_df.to_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_test_results.csv", index=False)
print(test_results_df)

joblib.dump(final_model, f"{ml_dir}/{label_tag}_{feature_name}_best_model_{best_model_name}.pkl")

prefix = f"{ml_dir}/{label_tag}_{feature_name}_"

plot_final_test_confusion_matrix(y_test, y_test_pred, best_model_name, cfg["label"], feature_name,
                                  save_path=prefix + "final_test_confusion_matrix")
plot_final_test_bar_chart(pd.DataFrame([test_metrics]), f"Final Test Performance — {cfg['label']} ({feature_name})",
                           save_path=prefix + "final_test_performance_barplot")
plot_final_test_roc(y_test, y_test_prob, best_model_name, cfg["label"], feature_name,
                     save_path=prefix + "final_test_ROC")

# --- LOPO ---
lopo_results_df = evaluate_lopo_host_disjoint(features_dir, best_model_name, final_model, feature_name)
lopo_results_df.to_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_lopo_host_disjoint_results.csv", index=False)
print(lopo_results_df)

if not lopo_results_df.empty:
    plot_lopo_results(lopo_results_df, f"Host-Disjoint LOPO Performance — {cfg['label']} ({feature_name})",
                       save_path=prefix + "LOPO_All_Metrics_barplot")
    plot_lopo_heatmap(lopo_results_df, f"Host-Disjoint LOPO Performance — {cfg['label']} ({feature_name})",
                       save_path=prefix + "LOPO_performance_heatmap")

print(f"\n{cfg['label']} complete: winning model = {best_model_name}")

#**Reuse a saved model checkpoint for LOPO**
This demonstrates how to reuse a saved model checkpoint for LOPO without needing to rerun hyperparameter tuning or keep the model in memory from an earlier session.

    IMPORTANT: the loaded model is used only as a source of hyperparameters,
    not as a fitted predictor. A fresh, unfitted clone is created and refit
    from scratch on each held-out species' own training data reusing the
    loaded model's existing fitted weights directly would mean it has already
    seen data from every species, defeating the purpose of testing
    generalization to an unseen pathogen.

####Step 1: Load the saved model

In [ ]:
import os
import joblib
import pandas as pd
from sklearn.base import clone

CURRENT_RATIO = 10        # <-- set to 1, 5, or 10 depending on which ratio you want
feature_name = "AAC"     # <-- must match the feature set the saved model was trained on

base_hpi_path = "/content/my_drive/MyDrive/HPI"
ratio_config = {
    1:  {"features_dir": f"{base_hpi_path}/Features/Balanced", "ml_dir": f"{base_hpi_path}/ML/Balanced", "label": "1:1"},
    5:  {"features_dir": f"{base_hpi_path}/Features/R5",       "ml_dir": f"{base_hpi_path}/ML/R5",       "label": "1:5"},
    10: {"features_dir": f"{base_hpi_path}/Features/R10",      "ml_dir": f"{base_hpi_path}/ML/R10",      "label": "1:10"},
}
cfg = ratio_config[CURRENT_RATIO]
features_dir = cfg["features_dir"]
ml_dir = cfg["ml_dir"]
label_tag = cfg["label"].replace(":", "_")

best_model_selection_df = pd.read_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv")
best_model_name = best_model_selection_df["Best_Model"].iloc[0]
saved_model_path = f"{ml_dir}/{label_tag}_{feature_name}_best_model_{best_model_name}.pkl"

print(f"Ratio: {cfg['label']} | Best model: {best_model_name}")
print(f"Model file: {saved_model_path}")

####Step 2: Use it as a hyperparameter template for LOPO

In [ ]:
from sklearn.base import clone
import joblib
import pandas as pd
import os

def evaluate_lopo_from_saved_model(features_dir, ml_dir, best_model_name, saved_model_path,
                                     feature_name, dataset_label, plot=True):
    """
    Loads a previously saved, fully-tuned model from disk, evaluates it using
    host-disjoint Leave-One-Pathogen-Out (LOPO) validation, and generates the
    same visualizations as the main pipeline (bar chart + heatmap).
    """
    print(f"Loading saved model from: {saved_model_path}")
    saved_model = joblib.load(saved_model_path)

    lopo_results = []
    folder = "lopo_folds_host_disjoint"
    pathogen_names = {
        "Bacillus": "bacillus", "Escherichia coli": "ecoli",
        "Francisella": "francisella", "Yersinia": "yersinia",
    }

    for species, file_species in pathogen_names.items():
        train_file = f"{features_dir}/{folder}/lopo_{file_species}_train_{feature_name}_features.csv"
        test_file = f"{features_dir}/{folder}/lopo_{file_species}_test_{feature_name}_features.csv"

        if not os.path.exists(train_file) or not os.path.exists(test_file):
            print(f"Missing file(s) for {species}, skipping.")
            continue

        lopo_train = pd.read_csv(train_file)
        lopo_test = pd.read_csv(test_file)
        X_tr, y_tr = get_X_y(lopo_train)
        X_te, y_te = get_X_y(lopo_test)

        lopo_model = clone(saved_model)
        lopo_model.fit(X_tr, y_tr)

        y_pred = lopo_model.predict(X_te)
        y_prob = lopo_model.predict_proba(X_te)[:, 1]
        metrics = calculate_metrics(y_te, y_pred, y_prob)

        lopo_results.append({"Model": best_model_name, "Held_Out_Pathogen": species, **metrics})
        print(f"{species}: MCC={metrics['MCC']:.3f}, AUC-ROC={metrics['AUC-ROC']:.3f}")

    lopo_results_df = pd.DataFrame(lopo_results)

    label_tag = dataset_label.replace(":", "_")
    lopo_results_df.to_csv(
        f"{ml_dir}/{label_tag}_{feature_name}_best_model_lopo_host_disjoint_results.csv",
        index=False
    )

    if plot and not lopo_results_df.empty:
        prefix = f"{ml_dir}/{label_tag}_{feature_name}_"
        plot_lopo_results(
            lopo_results_df,
            title=f"Host-Disjoint LOPO Performance — {dataset_label} ({feature_name})",
            save_path=prefix + "LOPO_All_Metrics_barplot"
        )
        plot_lopo_heatmap(
            lopo_results_df,
            title=f"Host-Disjoint LOPO Performance — {dataset_label} ({feature_name})",
            save_path=prefix + "LOPO_performance_heatmap"
        )
    elif plot:
        print("lopo_results_df is empty — nothing to plot.")

    return lopo_results_df

In [ ]:
best_model_selection_df = pd.read_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv")
best_model_name = best_model_selection_df["Best_Model"].iloc[0]
saved_model_path = f"{ml_dir}/{label_tag}_{feature_name}_best_model_{best_model_name}.pkl"

lopo_results_df = evaluate_lopo_from_saved_model(
    features_dir=features_dir,
    ml_dir=ml_dir,
    best_model_name=best_model_name,
    saved_model_path=saved_model_path,
    feature_name=feature_name,
    dataset_label=cfg["label"],
)

print(lopo_results_df)

##**Section 6: Deployment Model (Refit on Full Data)**

This section refits the selected best model on the complete dataset
(train_df + test_df combined) to produce a deployment-ready model for
predicting new, unlabeled host-pathogen pairs.

    IMPORTANT: this does NOT change or replace the final_model or
    test_metrics from Section 5.This section produces a SEPARATE model, saved
    under a distinct filename, used only for prediction on new pairs.

**This section is resumable independently if your session restarted**, you can jump straight here after rerunning Section 0 and Section 1,
as long as Section 5 has already been run at least once (so that best_model, train_df, and test_df exist, or can be reloaded below).


In [ ]:
# SECTION 6, STEP 1: RELOAD BEST MODEL AND DATA (skip if already in memory)
# =============================================================================

best_model_selection_df = pd.read_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv")
best_model_name = best_model_selection_df["Best_Model"].iloc[0]

tuned_model_path = f"{ml_dir}/{label_tag}_{feature_name}_tuned_{best_model_name}.pkl"
best_model = joblib.load(tuned_model_path)

train_df = pd.read_csv(f"{features_dir}/train_{feature_name}_split_features.csv")
test_df = pd.read_csv(f"{features_dir}/test_{feature_name}_split_features.csv")

print(f"Ratio: {cfg['label']} | Feature: {feature_name} | Best model: {best_model_name}")

In [ ]:
# SECTION 6, STEP 2: REFIT ON FULL DATA, SAVE DEPLOYMENT MODEL
# =============================================================================

from sklearn.base import clone

full_df = pd.concat([train_df, test_df], ignore_index=True)
X_full, y_full = get_X_y(full_df)

deployment_model = clone(best_model)   # fresh, unfitted clone with the same tuned hyperparameters
deployment_model.fit(X_full, y_full)

deployment_model_path = f"{ml_dir}/{label_tag}_{feature_name}_deployment_model_{best_model_name}.pkl"
joblib.dump(deployment_model, deployment_model_path)

print(f"Deployment model refit on {len(full_df)} pairs (train+hold-out combined) "
      f"and saved to {deployment_model_path}")



##**Section 7: Predicting New HP-PPI Pairs**

This section applies the saved deployment model (Section 6) to a new set of
host-pathogen protein pairs. It is fully resumable independently; if your
session restarted, you can jump straight here after rerunning Section 0 and
Section 1, without repeating tuning, final-test evaluation, or the
deployment refit.

**Prerequisite: The new pairs must already be preprocessed and feature  extracted using the SAME feature representation (feature_name) the
deployment model was trained on. No homology clustering is needed for new pairs; clustering is only used when constructing train/CV hold-out splits, not at inference time.**

In [ ]:
# SECTION 7, STEP 1: LOAD SAVED DEPLOYMENT MODEL
# =============================================================================

best_model_selection_df = pd.read_csv(f"{ml_dir}/{label_tag}_{feature_name}_best_model_selection.csv")
best_model_name = best_model_selection_df["Best_Model"].iloc[0]
deployment_model_path = f"{ml_dir}/{label_tag}_{feature_name}_deployment_model_{best_model_name}.pkl"

print(f"Ratio: {cfg['label']} | Feature: {feature_name} | Best model: {best_model_name}")
print(f"Loading deployment model from: {deployment_model_path}")

deployment_model = joblib.load(deployment_model_path)

In [ ]:
# SECTION 7, STEP 2: PREDICT NEW PAIRS
# =============================================================================

def predict_new_pairs(new_df, model, id_cols=("host_protein_id", "pathogen_protein_id")):
    """
    Predicts interaction status for new host-pathogen pairs.

    new_df must already be feature-extracted using the same descriptor
    (feature_name) the model was trained on. A dummy "label" column is
    added only because get_X_y() expects one, it is dropped immediately
    and never used in prediction.
    """
    X_new, _ = get_X_y(new_df.assign(label=0))

    y_pred = model.predict(X_new)
    y_prob = model.predict_proba(X_new)[:, 1]

    out_cols = [c for c in id_cols if c in new_df.columns]
    result = new_df[out_cols].copy() if out_cols else pd.DataFrame(index=new_df.index)
    result["predicted_label"] = y_pred            # 1 = predicted interaction, 0 = no interaction
    result["predicted_probability"] = y_prob       # model confidence, 0-1
    return result



In [ ]:
# Example usage:
# new_df = pd.read_csv(f"{features_dir}/new_pairs_{feature_name}_features.csv")
# new_predictions_df = predict_new_pairs(new_df, deployment_model)
# new_predictions_df.to_csv(f"{ml_dir}/{label_tag}_{feature_name}_new_ppi_predictions.csv", index=False)
# new_predictions_df.head()